# 07 — Hello-world in PyTorch

For six notebooks we wrote the forward pass, the backward pass, and the training loop **by hand**, in
plain numpy. That was the whole point: to know, with no mystery left, exactly what a neural network
computes. Now we hand **the same network as notebook 1** — the `2 → 16 → 3` classifier on `make_blobs`
— to **PyTorch**, the real modern tool. The one decisive change: **you stop writing the backward
pass.** You describe the forward computation, and PyTorch's *autograd* computes every gradient for you.

You will **read** this code, not write it from a blank cell — and the point of reading it is to
recognize every single line as something you already built. To prove the framework is not a black box,
we will check that its autograd gradient equals your by-hand gradient **to machine precision**.

**Prerequisites**
- Notebook 1 — the reference `2 → 16 → 3` net (forward, softmax + cross-entropy, the backward pass).
- Notebook 3 — backpropagation as the chain rule.
- Chapter 11 — gradient descent, epochs, mini-batches, the learning rate, the loss curve.

**What you'll be able to do**
- Define a network with `nn.Module` / `nn.Sequential` and read the **canonical training loop**.
- Explain **autograd** — why `loss.backward()` replaces the derivative work you did by hand.
- See torch's gradient match your by-hand gradient to ~`1e-17`, and train the same net to the same accuracy.
- Use the `.train()` / `.eval()` switch, and know when it will start to matter.

## Why a framework, now

Building everything by hand was the right way to *learn* — but it is not how real work is done. A
deep-learning framework earns its place for one decisive reason: **you stop writing the backward
pass.** You describe only the forward computation — the matrix multiplies, the activations, the loss —
and the framework records what you did and computes every gradient automatically. On a two-layer net
the backward pass is a page of algebra you can manage; on a hundred-layer net it is not.

PyTorch is the framework we use. It is not a new set of ideas — it is the ideas you already built,
made automatic and fast. We use **plain PyTorch**, nothing on top of it.

## The three things PyTorch gives you

1. **Tensors with autograd.** A torch tensor is like a numpy array, but it *remembers* the operations
   performed on it. When you call `loss.backward()`, torch walks that record backwards and fills in
   `∂loss/∂θ` for every parameter — the chain rule of notebook 3, executed for you.
2. **`nn.Module` (and `nn.Sequential`).** A tidy container that bundles a network's **parameters** with
   its **`forward`** computation, so a model is one object you can call, train, save, and move around.
3. **Optimizers** (`torch.optim.SGD`, and others). They perform the `θ ← θ − η·∇L` update you wrote by
   hand — one call, `optimizer.step()`.

One practical note before we start: neural-network training uses randomness (weight initialization,
shuffling), so results can vary run to run. To keep this notebook **reproducible**, we fix the seeds
and ask torch for deterministic CPU behaviour. With the four lines in the next cell, two separate runs
give bit-identical results on this machine.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.datasets import make_blobs
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from ml_course import colors, viz

viz.use_course_style()

SEED = 0
# The reproducibility contract (verified on this machine): these four lines make two separate runs
# produce bit-identical results on CPU.
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True)
torch.set_num_threads(1)
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())

# Notebook 1's data: a 3-class, 2-D make_blobs problem, split and standardized on the training set.
X, y = make_blobs(n_samples=300, centers=3, n_features=2, cluster_std=1.2, random_state=SEED)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEED)
scaler = StandardScaler().fit(X_tr)
X_tr, X_te = scaler.transform(X_tr), scaler.transform(X_te)
print(f"train {X_tr.shape}   test {X_te.shape}   classes {np.unique(y)}")

## The same net, in torch

Notebook 1's network was `2 → 16 → 3`: a linear map, a ReLU, a second linear map, then a softmax with
cross-entropy. In PyTorch that is three lines inside `nn.Sequential`:

```
nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 3))
```

Two things to notice. First, the softmax is **not** a layer here — it is folded into the loss:
`nn.CrossEntropyLoss` applies log-softmax and cross-entropy together, in one numerically-stable step,
so the model outputs raw scores (**logits**). Second, `nn.Linear` stores its weight with shape
`(out, in)` — the transpose of our `W` — and carries its own bias; apart from that layout, it computes
exactly the `X·W + b` you wrote by hand.

In [ ]:
net = nn.Sequential(
    nn.Linear(2, 16),   # input 2 -> hidden 16
    nn.ReLU(),          # the hidden activation
    nn.Linear(16, 3),   # hidden 16 -> 3 class logits
)
criterion = nn.CrossEntropyLoss()   # softmax + cross-entropy in one stable step

print(net)
for name, param in net.named_parameters():
    print(f"  {name:14} shape {tuple(param.shape)}")
print(f"  total parameters: {sum(p.numel() for p in net.parameters())}")

## Autograd: the backward pass you no longer write

In notebook 1 you derived the softmax + cross-entropy gradient `(p − y) / n` and pushed it back through
the ReLU and the two weight matrices by hand. PyTorch does this for you: as the forward pass runs, torch
records each operation on a graph; calling `loss.backward()` walks that graph in reverse and deposits
`∂loss/∂θ` into each parameter's `.grad`. Same chain rule, computed automatically.

Is it *really* the same computation, or a look-alike? We can settle it. We copy notebook 1's exact
weights into a torch network, run one forward and one `backward` on the same batch, and compare torch's
autograd gradient against our by-hand numpy gradient, number by number. (The check runs in **float64**
and writes the loss in its functional form, `F.cross_entropy` — the same softmax + cross-entropy as the
`nn.CrossEntropyLoss` above — so what we compare is the mathematics, not float32 rounding.)

In [ ]:
# Notebook 1's by-hand net (our reference), reproduced so we can compare gradients.
def init_params(scale=0.1, seed=SEED):
    rng = np.random.default_rng(seed)
    return {"W1": rng.standard_normal((2, 16)) * scale, "b1": np.zeros(16),
            "W2": rng.standard_normal((16, 3)) * scale, "b2": np.zeros(3)}


def np_softmax(z):
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)


def np_forward(p, X):
    z1 = X @ p["W1"] + p["b1"]
    h = np.maximum(0.0, z1)
    logits = h @ p["W2"] + p["b2"]
    return z1, h, logits, np_softmax(logits)


def np_backward(p, X, y):
    z1, h, logits, probs = np_forward(p, X)
    n = len(y)
    yoh = np.zeros_like(probs)
    yoh[np.arange(n), y] = 1.0
    d = (probs - yoh) / n                       # the softmax + cross-entropy gradient (notebook 1)
    dW2, db2 = h.T @ d, d.sum(0)
    dz1 = (d @ p["W2"].T) * (z1 > 0)            # backprop through ReLU
    dW1, db1 = X.T @ dz1, dz1.sum(0)
    return {"W1": dW1, "b1": db1, "W2": dW2, "b2": db2}


p = init_params()
g_np = np_backward(p, X_tr, y_tr)

# Copy those exact weights into a float64 torch net (nn.Linear weight is (out, in) = W.T), then let
# autograd compute the gradient on the same batch. float64 isolates the maths from float precision.
tnet = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 3)).double()
with torch.no_grad():
    tnet[0].weight.copy_(torch.tensor(p["W1"].T))
    tnet[0].bias.copy_(torch.tensor(p["b1"]))
    tnet[2].weight.copy_(torch.tensor(p["W2"].T))
    tnet[2].bias.copy_(torch.tensor(p["b2"]))

loss_t = F.cross_entropy(tnet(torch.tensor(X_tr)), torch.tensor(y_tr))
tnet.zero_grad()
loss_t.backward()

_, _, _, probs = np_forward(p, X_tr)
loss_np = -np.log(probs[np.arange(len(y_tr)), y_tr] + 1e-12).mean()
print(f"untrained loss:  by hand {loss_np:.6f}   torch {loss_t.item():.6f}   "
      f"(ln 3 = {np.log(3):.6f})")
print("max |autograd gradient - by-hand gradient|, per parameter:")
print(f"  W1 {np.abs(tnet[0].weight.grad.numpy().T - g_np['W1']).max():.1e}    "
      f"b1 {np.abs(tnet[0].bias.grad.numpy() - g_np['b1']).max():.1e}")
print(f"  W2 {np.abs(tnet[2].weight.grad.numpy().T - g_np['W2']).max():.1e}    "
      f"b2 {np.abs(tnet[2].bias.grad.numpy() - g_np['b2']).max():.1e}")

In [ ]:
fig, (axL, axR) = plt.subplots(1, 2, figsize=(13, 5))

# Left: each piece of the framework is a piece you already built.
axL.axis("off")
axL.set_xlim(0, 10)
axL.set_ylim(0, 10)
axL.set_title("What you built  ↔  what the framework does")
pairs = [
    ("your forward pass", "nn.Module.forward"),
    ("your by-hand backward", "loss.backward()  (autograd)"),
    ("your θ ← θ − η·∇L", "optimizer.step()"),
    ("your cross-entropy loss", "nn.CrossEntropyLoss"),
]
for i, (lhs, rhs) in enumerate(pairs):
    yy = 8.4 - i * 2.1
    axL.text(2.4, yy, lhs, ha="center", va="center", fontsize=10.5, color=colors.COLORS["text"],
             bbox=dict(boxstyle="round,pad=0.4", facecolor=colors.COLORS["muted"],
                       edgecolor=colors.COLORS["text"]))
    axL.annotate("", xy=(6.1, yy), xytext=(3.9, yy),
                 arrowprops=dict(arrowstyle="-|>", color=colors.COLORS["text"], lw=1.4))
    axL.text(7.8, yy, rhs, ha="center", va="center", fontsize=10.5, color=colors.COLORS["text"],
             bbox=dict(boxstyle="round,pad=0.4", facecolor=colors.COLORS["model"],
                       edgecolor=colors.COLORS["text"]))

# Right: the autograd gradient vs the by-hand gradient, every parameter on one plot.
g_torch = {"W1": tnet[0].weight.grad.numpy().T, "b1": tnet[0].bias.grad.numpy(),
           "W2": tnet[2].weight.grad.numpy().T, "b2": tnet[2].bias.grad.numpy()}
xs = np.concatenate([g_np[k].ravel() for k in ("W1", "b1", "W2", "b2")])
ys = np.concatenate([g_torch[k].ravel() for k in ("W1", "b1", "W2", "b2")])
lim = max(np.abs(xs).max(), np.abs(ys).max()) * 1.15
axR.plot([-lim, lim], [-lim, lim], color=colors.COLORS["muted"], lw=1, ls="--", zorder=0)
axR.scatter(xs, ys, s=32, color=colors.COLORS["model"],
            edgecolor=colors.COLORS["text"], linewidth=0.4)
maxd = max(np.abs(g_torch[k] - g_np[k]).max() for k in g_np)
axR.set_xlabel("by-hand gradient (numpy)")
axR.set_ylabel("autograd gradient (torch)")
axR.set_title(f"Every gradient identical  (max |Δ| ≈ {maxd:.0e})")
fig.tight_layout()
plt.show()

**Read the figure.** On the left, the correspondence: each thing you wrote by hand has an exact
counterpart in the framework — your forward pass is `nn.Module.forward`, your hand-derived backward is
`loss.backward()` (autograd), your weight update is `optimizer.step()`, your loss is the criterion. On
the right, every one of the network's gradient values is plotted with the by-hand value on the x-axis
and torch's autograd value on the y-axis: **all the points lie exactly on the diagonal**, and the
largest disagreement is around `1e-17` — machine precision. Autograd is not approximating your gradient
or replacing your understanding; it is computing the *same* gradient, automatically.

## The canonical training loop

Almost every PyTorch training loop is the same five steps, and you have written each one by hand:

```
optimizer.zero_grad()          # clear the previous step's gradients
output = model(X)              # forward pass
loss   = criterion(output, y)  # measure the error
loss.backward()               # autograd fills every .grad
optimizer.step()              # theta <- theta - eta * grad
```

Read that shape until it is automatic — it is the heartbeat of all deep-learning code. We run it now on
the same net and the same data as notebook 1.

In [ ]:
# Start from notebook 1's exact initial weights, so the comparison with the by-hand net is fair.
def make_net():
    m = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 3))
    with torch.no_grad():
        m[0].weight.copy_(torch.tensor(p["W1"].T, dtype=torch.float32))
        m[0].bias.copy_(torch.tensor(p["b1"], dtype=torch.float32))
        m[2].weight.copy_(torch.tensor(p["W2"].T, dtype=torch.float32))
        m[2].bias.copy_(torch.tensor(p["b2"], dtype=torch.float32))
    return m


X_tr_t = torch.tensor(X_tr, dtype=torch.float32)
y_tr_t = torch.tensor(y_tr)
model = make_net()
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)

torch_losses = []
for _ in range(400):
    optimizer.zero_grad()             # clear last step's gradients
    logits = model(X_tr_t)            # forward
    loss = criterion(logits, y_tr_t)  # loss
    loss.backward()                   # autograd fills every .grad
    optimizer.step()                  # theta <- theta - eta * grad
    torch_losses.append(loss.item())


def accuracy(model, Xa, ya):
    with torch.no_grad():
        pred = model(torch.tensor(Xa, dtype=torch.float32)).argmax(1).numpy()
    return (pred == ya).mean()


print(f"final loss {torch_losses[-1]:.4f}   "
      f"train acc {accuracy(model, X_tr, y_tr):.3f}   test acc {accuracy(model, X_te, y_te):.3f}")

In [ ]:
# Train notebook 1's by-hand numpy net the same way (full-batch GD, lr 0.5), and overlay the curves.
p_np = init_params()
np_losses = []
for _ in range(400):
    _, _, _, probs = np_forward(p_np, X_tr)
    np_losses.append(-np.log(probs[np.arange(len(y_tr)), y_tr] + 1e-12).mean())
    g = np_backward(p_np, X_tr, y_tr)
    for k in p_np:
        p_np[k] -= 0.5 * g[k]

fig, ax = plt.subplots(figsize=(8.5, 5))
ax.plot(np_losses, color=colors.COLORS["train"], lw=3, label="by hand (numpy)")
ax.plot(torch_losses, color=colors.COLORS["model"], lw=1.5, ls="--", label="PyTorch (autograd)")
ax.axhline(np.log(3), color=colors.COLORS["muted"], lw=1, ls=":", label="ln 3 = chance")
ax.set_xlabel("epoch")
ax.set_ylabel("training loss")
ax.set_title("The same net, two implementations: one loss curve")
ax.legend()
fig.tight_layout()
plt.show()

**Read the figure.** Two loss curves are plotted — the by-hand numpy net (solid) and the PyTorch net
(dashed) — and they lie on top of each other, both falling from `ln 3 ≈ 1.10` (a network that knows
nothing spreads its guesses evenly across the three classes — the small random weights nudge the start
a hair above the exact `ln 3 = 1.099`, as the printout earlier showed) down to about `0.31`. The torch net reaches
**train 0.867 / test 0.893** — the very numbers notebook 1 reported, because it is the same network, the
same data, and the same starting weights. PyTorch changed *how we express* the computation, not *what*
it computes.

## `.train()` and `.eval()` — the mode switch

One idea here is genuinely new. A torch module has two modes, set with `model.train()` and
`model.eval()`. For the plain network above they behave identically — but the moment we add **dropout**
or **batch normalization** (notebook 8) the modes diverge: dropout is active only in training, and batch
norm uses the current batch's statistics while training but the **running statistics** (notebook 6) at
evaluation. Flipping the switch is how you tell the network "we are learning now" versus "we are
predicting now." We meet it here, on a net where it changes nothing, so it is already familiar when it
starts to matter.

In [ ]:
model.train()
print("after .train():  model.training =", model.training)
model.eval()
print("after .eval():   model.training =", model.training)

# For this net (no dropout, no batch norm), the two modes give identical predictions:
with torch.no_grad():
    model.train()
    out_train = model(X_tr_t)
    model.eval()
    out_eval = model(X_tr_t)
print("predictions identical in train vs eval here:", torch.equal(out_train, out_eval))

## What carried over, what's new

Nothing about the *ideas* changed. Epochs, mini-batches, the learning rate, and reading a loss curve are
exactly as you learned them in chapter 11. What is new is **tooling**:

- **autograd** — you describe the forward pass and stop writing the backward one;
- the **`nn.Module`** idiom — parameters and `forward` bundled into one object;
- **`.train()` / `.eval()`** modes — which will matter the moment dropout and batch norm arrive.

That is the whole framework move: a better set of tools for the ideas you already own.

## What you built

- The same `2 → 16 → 3` network as notebook 1, now in PyTorch: `nn.Sequential` + `nn.CrossEntropyLoss`.
- The **canonical training loop** — `zero_grad → forward → loss → backward → step` — read line by line.
- A proof, to ~`1e-17`, that **autograd's gradient equals the gradient you derived by hand** — the
  framework is exactly what you built, automated.
- The same train/test accuracy as the by-hand net, and the `.train()` / `.eval()` switch.

You can now open any PyTorch training script and recognize every line in it.

## Where this goes next

**Notebook 8** turns the model's knobs in torch: real `nn.Dropout` and `nn.BatchNorm` (the layers you
built by hand in notebooks 5 and 6, now one line each), weight initialization via `torch.nn.init`
(He / Xavier, notebook 4), and the optimizer menu — SGD, momentum, Adam. It is also where `.train()` and
`.eval()` stop being a formality and start to change what the network does.

## Your turn

Each task **modifies the loop you read above** — you never write the backward pass.

1. **(warm-up)** Swap `torch.optim.SGD` for `torch.optim.Adam` (keep `lr=0.5`, then try `lr=0.05`) and
   compare the loss curve. Which optimizer settles faster here?
2. **(core)** Add a second hidden layer — `nn.Linear(16, 16), nn.ReLU()` — to the `nn.Sequential` and
   retrain. Does the extra depth help on this easy 2-D problem, or is 16 hidden units already enough?
3. **(reach)** Re-initialize the network with He initialization: `torch.nn.init.kaiming_normal_` on each
   `Linear` weight (notebook 4). Then re-run the gradient-match from earlier with torch's *default* init
   and confirm the autograd gradient still equals a fresh by-hand gradient — autograd is exact whatever
   the weights are.

## References

- Paszke, A., Gross, S., Massa, F., et al. (2019). PyTorch: An Imperative Style, High-Performance Deep
  Learning Library. *Advances in Neural Information Processing Systems 32* (NeurIPS).
- PyTorch documentation: https://pytorch.org/docs/stable/ (autograd, `nn.Module`, `torch.optim`).

*Previous:* **12.6 — normalization.**  *Next:* **12.8 — the model and its parameters in PyTorch.**